In [ ]:
# IMPORTS
# _________________________________________________________________________
# Core libraries for GNN training and evaluation

!pip -q install torch-geometric
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINEConv, global_add_pool, BatchNorm

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import random

seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
# HeuGNetData-triplet-processing__________________________________________
# Required so PyG's DataLoader correctly shifts triplet node indices when
# batching multiple graphs together (without this, batch_size > 1 silently
# corrupts all triplet messages).

from torch_geometric.data import Data

class HeuGNetData(Data):
    def __inc__(self, key, value, *args, **kwargs):
        if key == 'triplets':
            return self.num_nodes
        return super().__inc__(key, value, *args, **kwargs)


def convert_to_heugndata(data_list):

    converted = []
    for g in data_list:
        if g is None:
            continue
        new_g = HeuGNetData(
            x          = g.x,
            edge_index = g.edge_index,
            edge_attr  = g.edge_attr,
            y          = g.y,
            triplets   = g.triplets,
            angles     = g.angles,
        )
        converted.append(new_g)
    return converted



In [ ]:
# DATA LOADING
# _________________________________________________________________________
import torch

graph_list = torch.load(
    "graphs_enhanced.pt",
    map_location="cpu",
    weights_only=False
)

graph_list = [g for g in graph_list if g is not None]

print("Total graphs:", len(graph_list))
print(graph_list[0])

In [ ]:

graph_list = convert_to_heugndata(graph_list)

In [ ]:
# Verify required attributes
#____________________________________________

sample = graph_list[0]

print("Has x:", hasattr(sample, "x"))
print("Has edge_index:", hasattr(sample, "edge_index"))
print("Has edge_attr:", hasattr(sample, "edge_attr"))
print("Has triplets:", hasattr(sample, "triplets"))
print("Has angles:", hasattr(sample, "angles"))
print("Has y:", hasattr(sample, "y"))

In [ ]:
# TARGET NORMALIZATION
#____________________________________________

y_vals = torch.tensor([g.y.item() for g in graph_list])

mean = y_vals.mean()
std = y_vals.std()

for g in graph_list:
    g.y = (g.y - mean) / std

In [ ]:
# DATA SPLITTING
#____________________________________________
# Splitting dataset into train, validation, and test (80 / 10 / 10)

train_data, temp_data = train_test_split(
    graph_list, test_size=0.20, random_state=42
)

val_data, test_data = train_test_split(
    temp_data, test_size=0.50, random_state=42
)

print("Train:", len(train_data))
print("Val:  ", len(val_data))
print("Test: ", len(test_data))

In [ ]:
# DATALOADERS
#____________________________________________
# Preparing batched loaders

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_data, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_data, batch_size=32, shuffle=False)

In [ ]:
# LOSS FUNCTION
# _______________________________________________________________

loss_fn = nn.SmoothL1Loss(beta=0.5)

In [ ]:
# RBF EMBEDDING
# _______________________________________________________________

class RBFEmbedding(nn.Module):
    def __init__(self, num_radial=32):
        super().__init__()
        self.centers = torch.linspace(0, 5, num_radial)
        self.gamma = 15.0

    def forward(self, d):
        d = d.view(-1, 1)
        centers = self.centers.to(d.device)
        return torch.exp(-self.gamma * (d - centers)**2)

In [ ]:
# MODEL DEFINITION
# _______________________________________________________________

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GINEConv, global_add_pool, BatchNorm

class HybridGNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.rbf = RBFEmbedding(32)

        self.edge_mlp = nn.Sequential(
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, 64)
        )

        self.angle_mlp = nn.Sequential(
            nn.Linear(1, 32),
            nn.ReLU(),
            nn.Linear(32, 64)
        )

        def mlp(in_dim, out_dim):
            return nn.Sequential(
                nn.Linear(in_dim, out_dim),
                nn.ReLU(),
                nn.Linear(out_dim, out_dim)
            )

        self.conv1 = GINEConv(mlp(5, 128), edge_dim=64)
        self.bn1 = BatchNorm(128)

        self.conv2 = GINEConv(mlp(128, 128), edge_dim=64)
        self.bn2 = BatchNorm(128)

        self.conv3 = GINEConv(mlp(128, 256), edge_dim=64)
        self.bn3 = BatchNorm(256)

        self.conv4 = GINEConv(mlp(256, 128), edge_dim=64)
        self.bn4 = BatchNorm(128)

        self.fc1 = nn.Linear(128, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 32)

        self.fc4 = nn.Sequential(
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

        self.dropout = nn.Dropout(0.1)

        self.angle_weight = nn.Parameter(torch.tensor(0.5))

        # weight init
        def init_weights(m):
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

        self.apply(init_weights)

    def forward(self, data):
        x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch

        edge_attr = self.rbf(edge_attr)
        edge_attr = self.edge_mlp(edge_attr)

        if hasattr(data, "triplets") and hasattr(data, "angles"):
            theta = data.angles / torch.pi
            angle_feat = self.angle_mlp(theta)

            j = data.triplets[:, 1]
            angle_msg = torch.zeros(x.size(0), 64, device=x.device)
            angle_msg.index_add_(0, j, angle_feat)

            counts = torch.bincount(j, minlength=x.size(0)).unsqueeze(1) + 1e-6
            angle_msg = angle_msg / counts

            x = x + 1.5 * torch.tanh(self.angle_weight) * angle_msg[:, :x.size(1)]

        x1 = F.relu(self.bn1(self.conv1(x, edge_index, edge_attr)))
        x2 = F.relu(self.bn2(self.conv2(x1, edge_index, edge_attr)))
        x3 = F.relu(self.bn3(self.conv3(x2 + x1, edge_index, edge_attr)))
        x4 = F.relu(self.bn4(self.conv4(x3, edge_index, edge_attr)))

        x = global_add_pool(x4, batch)

        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))

        return self.fc4(x)

In [ ]:
# TRAIN FUNCTION
# _______________________________________________________________

def train_model(model, optimizer, epochs=100):

    best_val_loss = float('inf')
    best_state    = None

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max   = 100,
        eta_min = 1e-5
    )

    train_losses, val_losses = [], []

    for epoch in range(epochs):

        model.train()
        total_loss = 0

        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()

            pred = model(batch).view(-1)
            loss = loss_fn(pred, batch.y.view(-1))

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            optimizer.step()

            total_loss += loss.item()

        train_loss = total_loss / len(train_loader)

        model.eval()
        val_loss = 0

        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(device)
                pred  = model(batch).view(-1)
                loss  = loss_fn(pred, batch.y.view(-1))
                val_loss += loss.item()

        val_loss /= len(val_loader)
        scheduler.step()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state    = model.state_dict()

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        print(f"Epoch {epoch+1:03d} | Train: {train_loss:.4f} | Val: {val_loss:.4f}")

    print(f"\nBest Val Loss: {best_val_loss:.4f}")
    return train_losses, val_losses, best_state

In [ ]:
# TRAINING
# _______________________________________________________________

model = HybridGNN().to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=6e-4,
    weight_decay=1e-4
)

train_losses, val_losses, best_state = train_model(model, optimizer)

# loading best model
model.load_state_dict(best_state)

# storing final losses for table
final_train_loss = train_losses[-1]
final_val_loss = val_losses[-1]

In [ ]:
# EVALUATE
#__________________________________________

model.eval()
all_preds, all_actuals = [], []

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        pred  = model(batch).view(-1)
        all_preds.extend(pred.cpu().numpy())
        all_actuals.extend(batch.y.view(-1).cpu().numpy())

y_pred = np.array(all_preds)
y_true = np.array(all_actuals)

mae  = mean_absolute_error(y_true, y_pred)
mse  = mean_squared_error(y_true, y_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_true, y_pred)

print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")


print("\n=== RESULTS ===")
print(f"MAE  : {mae:.4f}")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")


In [ ]:
# PLOTS
#__________________________________________

import matplotlib.pyplot as plt
import os

C_MAIN  = "#1D9E75"
C_DARK  = "#085041"
C_LIGHT = "#9FE1CB"
C_ALPHA = 0.72

SAVE_DIR = "/content/Transformed_Output"
os.makedirs(SAVE_DIR, exist_ok=True)

residuals = y_true - y_pred
lims      = [min(y_true.min(), y_pred.min()),
             max(y_true.max(), y_pred.max())]

def apply_style(ax, xlabel, ylabel, title):
    ax.set_facecolor("white")
    ax.set_xlabel(xlabel, fontsize=13, labelpad=6)
    ax.set_ylabel(ylabel, fontsize=13, labelpad=6)
    ax.set_title(title, fontsize=14, fontweight="bold", pad=10)
    ax.tick_params(labelsize=11)
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color("#cccccc")


# PLOT 2 — Predicted vs Actual
#_________________________________________________________________
fig, ax = plt.subplots(figsize=(6, 5), facecolor="white")
ax.scatter(y_true, y_pred,
           color=C_MAIN, alpha=C_ALPHA, s=30, edgecolors="none",
           label="Predictions")
ax.plot(lims, lims, color=C_DARK, linewidth=1.8,
        linestyle="--", label="Ideal fit")
apply_style(ax,
    xlabel="Actual value (eV)",
    ylabel="Predicted value (eV)",
    title="Predicted vs Actual")
ax.legend(fontsize=11, framealpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "plot2_pred_vs_actual.png"),
            dpi=600, bbox_inches="tight", facecolor="white")
plt.show()
print("Saved: plot2_pred_vs_actual.png")


# PLOT 3 — Residual Plot
#_________________________________________________________________
fig, ax = plt.subplots(figsize=(6, 5), facecolor="white")
ax.scatter(y_pred, residuals,
           color=C_MAIN, alpha=C_ALPHA, s=30, edgecolors="none")
ax.axhline(0, color=C_DARK, linewidth=1.8, linestyle="--",
           label="Zero residual")
apply_style(ax,
    xlabel="Predicted value (eV)",
    ylabel="Residual (eV)",
    title="Residual Plot")
ax.legend(fontsize=11, framealpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "plot3_residuals.png"),
            dpi=600, bbox_inches="tight", facecolor="white")
plt.show()
print("Saved: plot3_residuals.png")


# PLOT 4 — Error Distribution
#_________________________________________________________________
fig, ax = plt.subplots(figsize=(6, 5), facecolor="white")
ax.hist(residuals, bins=40,
        color=C_MAIN, edgecolor=C_LIGHT, linewidth=0.6, alpha=0.88)
ax.axvline(0, color=C_DARK, linewidth=1.8, linestyle="--", label="Zero error")
apply_style(ax,
    xlabel="Residual (eV)",
    ylabel="Count",
    title="Error Distribution")
ax.legend(fontsize=11, framealpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "plot4_error_dist.png"),
            dpi=600, bbox_inches="tight", facecolor="white")
plt.show()
print("Saved: plot4_error_dist.png")


# ROW FIGURE — Plots 2, 3, 4
#____________________________________________


# For adjusting with reference to Model 3's xlim
HIST_XLIM = (-1.29, 1.29)

fig, axes = plt.subplots(1, 3, figsize=(18, 5), facecolor="white")

ax = axes[0]
ax.scatter(y_true, y_pred,
           color=C_MAIN, alpha=C_ALPHA, s=30, edgecolors="none",
           label="Predictions")
ax.plot(lims, lims, color=C_DARK, linewidth=1.8,
        linestyle="--", label="Ideal fit")
apply_style(ax,
    xlabel="Actual value (eV)",
    ylabel="Predicted value (eV)",
    title="Predicted vs Actual")
ax.legend(fontsize=10, framealpha=0.5)

ax = axes[1]
ax.scatter(y_pred, residuals,
           color=C_MAIN, alpha=C_ALPHA, s=30, edgecolors="none")
ax.axhline(0, color=C_DARK, linewidth=1.8, linestyle="--",
           label="Zero residual")
apply_style(ax,
    xlabel="Predicted value (eV)",
    ylabel="Residual (eV)",
    title="Residual Plot")
ax.legend(fontsize=10, framealpha=0.5)

ax = axes[2]
ax.hist(residuals, bins=40,
        color=C_MAIN, edgecolor=C_LIGHT, linewidth=0.6, alpha=0.88)
ax.axvline(0, color=C_DARK, linewidth=1.8, linestyle="--", label="Zero error")
if HIST_XLIM:
    ax.set_xlim(HIST_XLIM)
apply_style(ax,
    xlabel="Residual (eV)",
    ylabel="Count",
    title="Error Distribution")
ax.legend(fontsize=10, framealpha=0.5)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "Transformed_row.png"),
            dpi=600, bbox_inches="tight", facecolor="white")
plt.show()
print("Saved: Transformed_row.png")

In [ ]:
# SAVE MODEL + WEIGHTS + RESULTS
#____________________________________________

import os
import torch

SAVE_DIR = "Transformed_Output"
os.makedirs(SAVE_DIR, exist_ok=True)


# Full checkpoint


torch.save(
    {
        "model_state": model.state_dict(),

        "mae": mae,
        "mse": mse,
        "rmse": rmse,



        "norm_mean": mean.item(),
        "norm_std": std.item()
    },
    os.path.join(
        SAVE_DIR,
        "Transformed_final.pth"
    )
)


# Weights only


torch.save(
    model.state_dict(),
    os.path.join(
        SAVE_DIR,
        "Transformed_weights.pth"
    )
)


# Results


results_text = f"""
====================================
Transformed Model Results
====================================

MAE         : {mae:.6f}
MSE         : {mse:.6f}
RMSE        : {rmse:.6f}



Train Size  : {len(train_data)}
Val Size    : {len(val_data)}
Test Size   : {len(test_data)}

Normalization Mean : {mean.item():.6f}
Normalization Std  : {std.item():.6f}

Total Graphs : {len(graph_list)}
"""

with open(
    os.path.join(
        SAVE_DIR,
        "Transformed_results.txt"
    ),
    "w"
) as f:
    f.write(results_text)

print(results_text)

print("Saved Transformed_final.pth")
print("Saved Transformed_weights.pth")
print("Saved Transformed_results.txt")

In [ ]:
import shutil

shutil.make_archive(
    "Transformed_Output",
    "zip",
    "Transformed_Output"
)

print("Created: Transformed_Output.zip")

In [ ]:
import os

for f in sorted(os.listdir("Transformed_Output")):
    print(f)